# 02 — Label integrity and leakage audit

**Purpose:** examine the target definition, class distribution, and suspiciously target-aligned fields.

This notebook treats `anomaly` as the candidate target because that is the target column observed in the supplied CSV. It does **not** assume that the labels are independently or correctly generated.

## 1. Load data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Put the CSV beside this notebook, or change DATA_PATH to its full path.
DATA_PATH = Path("metaverse_transactions_dataset(1).csv")

if not DATA_PATH.exists():
    candidates = list(Path(".").glob("*.csv"))
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Place the original CSV beside this notebook "
        f"or edit DATA_PATH. CSV files in this folder: {[p.name for p in candidates]}"
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

## 2. Target distribution and basic validity

In [ ]:
TARGET = "anomaly"
if TARGET not in df.columns:
    raise KeyError(f"Expected target column {TARGET!r}; found {df.columns.tolist()}")

print("Target missing:", int(df[TARGET].isna().sum()))
display(df[TARGET].value_counts(dropna=False).rename_axis(TARGET).to_frame("count"))
display((df[TARGET].value_counts(normalize=True, dropna=False) * 100).round(3).rename("percent").to_frame())

## 3. Numeric feature versus target summaries

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != TARGET]
for col in numeric_cols:
    print(f"\\n### {col}")
    display(df.groupby(TARGET, dropna=False)[col].agg(["count","min","median","mean","max","std"]))

## 4. Categorical feature versus target cross-tabs

In [ ]:
categorical_cols = [c for c in df.columns if c != TARGET and
                    (df[c].dtype == "object" or df[c].nunique(dropna=False) <= 30)]
for col in categorical_cols:
    print(f"\\n### {col}")
    ct = pd.crosstab(df[col].fillna("<MISSING>").astype(str),
                     df[TARGET].fillna("<MISSING>").astype(str),
                     normalize="index")
    display(ct.round(3))

## 5. Investigate the previously flagged fields

In [ ]:
for col in ["risk_score", "transaction_type", "purchase_pattern", "age_group"]:
    if col not in df.columns:
        continue
    print(f"\\n### {col}")
    if pd.api.types.is_numeric_dtype(df[col]):
        display(df.groupby(TARGET)[col].agg(["min","max","mean","median","nunique"]))
        ranges = df.groupby(TARGET)[col].agg(["min","max"])
        print("Class range overlap must be assessed from these intervals.")
    else:
        display(pd.crosstab(df[col], df[TARGET], margins=True))

## 6. Single-feature predictability screen (diagnostic only)

In [ ]:
# This is a diagnostic screen, not proof of leakage. High predictability can be
# legitimate, target-derived, or caused by synthetic label construction.
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(str)
if y.nunique() == 2:
    y_enc = LabelEncoder().fit_transform(y)
    scoring = "roc_auc"
else:
    y_enc = LabelEncoder().fit_transform(y)
    scoring = "roc_auc_ovr"

rows = []
for col in X.columns:
    xcol = X[[col]].copy()
    if pd.api.types.is_numeric_dtype(xcol[col]):
        prep = ColumnTransformer([("num", SimpleImputer(strategy="median"), [col])])
    else:
        prep = ColumnTransformer([("cat", make_pipeline(
            SimpleImputer(strategy="most_frequent"),
            OneHotEncoder(handle_unknown="ignore")), [col])])
    model = make_pipeline(prep, RandomForestClassifier(
        n_estimators=150, min_samples_leaf=5, class_weight="balanced",
        random_state=42, n_jobs=-1))
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    try:
        scores = cross_val_score(model, xcol, y_enc, cv=cv, scoring=scoring, n_jobs=-1)
        rows.append({"feature": col, "mean_cv_score": scores.mean(), "std_cv_score": scores.std()})
    except Exception as e:
        rows.append({"feature": col, "mean_cv_score": np.nan, "std_cv_score": np.nan, "error": str(e)})

screen = pd.DataFrame(rows).sort_values("mean_cv_score", ascending=False)
display(screen)

## 7. Leakage decision register

Fill this table after tracing how each field was created and when it became available.

| Field | Available at prediction time? | Derived from target/risk rule? | Decision | Evidence |
|---|---|---|---|---|
| risk_score | Unknown | Suspected | Hold out pending provenance | Perfect class separation observed in initial inspection |
| transaction_type | Unknown | Suspected proxy | Investigate | Strong class-specific pattern |
| purchase_pattern | Unknown | Unknown | Investigate | Check construction logic |
| age_group | Unknown | Unknown | Investigate | Check whether synthetic rules assigned labels by age group |

**Do not interpret a high cross-validation score as proof of genuine predictive power.** A feature may leak the target or encode the label-generation rule.

## Interpretation checklist

1. Obtain the label-generation code or written rules.
2. Determine the exact moment each feature becomes available.
3. Establish whether `risk_score` is an input, an output, or a transformation of the target rule.
4. Separate direct leakage, proxy leakage, and legitimate predictive association.
5. Do not finalize the feature list until provenance is documented.